# Notebook 08 — End-to-End Evaluation and Ablations
**CliniVer** · Experiment `EXP-08-001`

| | |
|---|---|
| **Objective** | Evaluate the complete pipeline (claim → retrieval → verification → calibration → second opinion → abstention), attribute its errors to components, ablate each design decision, and fix the final configuration |
| **Research question** | **RQ7:** How do errors propagate end-to-end? Which component contributes most? |
| **Hypothesis (pre-registered)** | **H7:** Retrieval failure accounts for a larger share of end-to-end errors than verifier error |
| **Decision rules** | `configs/e2e.yaml` (fixed before running): H7 attribution rule; final-selection rule |
| **Labels** | Benchmark (SciFact, human), synthetic (NB04 context items) and **silver** (reference statements). **Real-claim gold labels are not used** (owner decision 2026-11-21, D7): real-claim accuracy is NOT EVALUATED until clinical validation |

## 1. Method
- **End-to-end benchmark:** the 300 SciFact dev claims whose evidence is unanimous (SUP / CON) or absent (INS); claims with mixed evidence are excluded. Evidence is retrieved over the NB05 primary corpus (5,183 SciFact + 124,883 CliniVer abstracts).
- **Pipeline (ADR-0011 … ADR-0014):**
  1. Hybrid BM25 + MedCPT (RRF), MedCPT cross-encoder over the top 50.
  2. Top-1 evidence.
  3. Fine-tuned NLI.
  4. Temperature scaling and the multi-signal confidence (fitted on HealthVer dev).
  5. Qwen3-8B second opinion below the 25th percentile.
  6. ABSTAINED when the two verifiers disagree.
- **Metrics:**
  - macro-F1 (SUP / CON / INS) on the answered claims, coverage, false-support rate (FSR) on the answered claims;
  - **end-to-end macro-F1 with abstentions counted as errors** (a conservative single number);
  - latency per claim.
- **H7 attribution** (pipeline without abstention): an error that disappears when the retrieved top-1 evidence is replaced by **oracle evidence** (a gold evidence abstract; for INS claims, a cited abstract) is *retrieval-attributable*. An error that remains is *verifier-attributable*.
- **Ablations:** retrieval (BM25 only, MedCPT only, hybrid without re-ranker, hybrid with BGE-base, weighted hybrid); evidence (top-3 naive aggregation); verifier (Qwen3-8B only, zero-shot NLI); confidence (uncalibrated, no second opinion, no abstention); contextualization on the NB04 synthetic context-dependent items (claim only, question + claim, rewrite F; oracle = original claim).
- **Silver check of the real claims:** each real claim is compared with its question's reference statements by the NB03 NLI judge. Silver SUP = some statement entails the claim; silver CON = some statement contradicts it.

## 2. Setup

In [ ]:
import json, os, random, re, time, collections, io, tarfile, gc
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, yaml
from IPython.display import display

from cliniver.config import load_project_config
from cliniver.config.llm_config import load_llm_config
from cliniver.utils.seed import set_seed
from cliniver.utils.experiment import Experiment
from cliniver.ingestion.http import RateLimitedClient
from cliniver.ingestion.scifact import parse_release
from cliniver.retrieval.encoders import Encoder
from cliniver.retrieval.sparse import BM25
from cliniver.retrieval.fusion import rrf, weighted
from cliniver.retrieval.pipeline import scifact_primary_corpus
from cliniver.verification.datasets import load_sets
from cliniver.verification.verifiers import NLIVerifier, LLMVerifier
from cliniver.verification.confidence import ConfidenceModel, LAB
from cliniver.llm.runtime import ManagedClient
from sentence_transformers import CrossEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pcfg = load_project_config(); lcfg = load_llm_config()
ROOT = pcfg.paths.absolute("data_raw").parents[1]
cfg = yaml.safe_load((ROOT / "configs" / "e2e.yaml").read_text(encoding="utf-8"))
vcfg = yaml.safe_load((ROOT / "configs" / "verification.yaml").read_text(encoding="utf-8"))
rcfg = yaml.safe_load((ROOT / "configs" / "retrieval.yaml").read_text(encoding="utf-8"))
ccfg = yaml.safe_load((ROOT / "configs" / "context.yaml").read_text(encoding="utf-8"))
SEED = set_seed(cfg["seed"]); SNAP = cfg["snapshot"]
SMOKE = os.environ.get("CLINIVER_SMOKE") == "1"
RAW = pcfg.paths.absolute("data_raw"); PROC = pcfg.paths.absolute("data_processed") / SNAP
INTERIM = pcfg.paths.absolute("data_interim") / SNAP; EMB = INTERIM / "embeddings"; LLM_LOG = INTERIM / "llm_calls"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 110)
exp = Experiment(exp_id="EXP-08-000-smoke" if SMOKE else "EXP-08-001", title="End-to-end evaluation and ablations", research_question="RQ7 / H7",
                 config={"e2e": cfg}, seed=SEED, packages=["torch", "transformers", "sentence-transformers", "bm25s", "scikit-learn", "pandas", "cliniver"])
FT = sorted(d for d in (INTERIM / "models").glob("nli_ft_*") if (d / "config.json").exists())[0]
nli = NLIVerifier(str(FT), device=DEV, model=AutoModelForSequenceClassification.from_pretrained(FT), tokenizer=AutoTokenizer.from_pretrained(FT),
                  max_length=vcfg["finetune"]["max_length"])
SETS, sf_raw = load_sets(vcfg, RAW, SNAP, RateLimitedClient("datasets", min_interval_s=1.0, max_in_flight=1), seed=SEED)
CM = ConfidenceModel.fit(nli, SETS["healthver_dev"])
_zc = nli.logits([x["evidence"] for x in SETS["healthver_dev"]], [x["claim"] for x in SETS["healthver_dev"]])
RAW_Q25 = float(np.quantile(np.exp(_zc).max(1) / np.exp(_zc).sum(1), 0.25))   # routing threshold on the UNCALIBRATED scale (ablation)
def managed(key):
    return ManagedClient(lcfg, key, LLM_LOG / f"{key}.jsonl", server_log=pcfg.paths.absolute("logs") / f"llama-server_nb08_{key}.log")
exp.log_result("confidence_model", {"temperature": CM.t, "route_threshold": CM.route_threshold, "abstain_threshold": CM.abstain_threshold})
print("verifier", FT.name, "| T", round(CM.t, 3), "| route <", round(CM.route_threshold, 3), "| smoke", SMOKE)

## 3. Benchmark claims and retrieval variants

In [ ]:
DOC_IDS, TITLES, ABSTRACTS, N_SF = scifact_primary_corpus(sf_raw, PROC / "documents.jsonl")
DOC_TEXT = [f"{t}. {a}" for t, a in zip(TITLES, ABSTRACTS)]
sf_corpus, _ = parse_release(sf_raw)
tf = tarfile.open(fileobj=io.BytesIO(sf_raw), mode="r:gz")
dev = [json.loads(l) for l in tf.extractfile(next(x for x in tf.getnames() if x.endswith("claims_dev.jsonl") and "/._" not in x and "cross" not in x)).read().decode().splitlines() if l.strip()]
CLAIMS = []
for c in dev:
    labs = {s["label"] for d in (c.get("evidence") or {}).values() for s in d}
    g = "SUP" if labs == {"SUPPORT"} else "CON" if labs == {"CONTRADICT"} else "INS" if not labs else None
    if g is None: continue
    ev_docs = [d for d in (c.get("evidence") or {}) if d in sf_corpus]
    cited = [str(d) for d in c.get("cited_doc_ids", []) if str(d) in sf_corpus]
    oracle = ev_docs[0] if ev_docs else (cited[0] if cited else None)
    CLAIMS.append({"id": c["id"], "claim": c["claim"], "gold": g, "oracle_text": sf_corpus[oracle] if oracle else ""})
if SMOKE: CLAIMS = CLAIMS[:40]
Q = [c["claim"] for c in CLAIMS]; GOLD = np.array([c["gold"] for c in CLAIMS])
print(len(CLAIMS), "claims", Counter(GOLD))
DEPTH = rcfg["first_stage_depth"]
bm = BM25(DOC_TEXT, variant="default"); BM = bm.search(Q, DEPTH)
dense = {}
for key in ("medcpt", "bge-base"):
    m = rcfg["encoders"][key]
    enc = Encoder(key, m["repo"], m["revision"], m.get("query_prefix", ""), m.get("doc_prefix", ""), m.get("doc_repo"), m.get("doc_revision"), device=DEV)
    D = torch.from_numpy(np.load(EMB / f"{key}_primary.npy")).to(DEV)
    dense[key] = Encoder.search(enc.encode_queries(Q), D, DEPTH)
    enc.unload(); del D; torch.cuda.empty_cache()
ce_ = rcfg["rerankers"]["medcpt-ce"]; CE = CrossEncoder(ce_["repo"], revision=ce_["revision"], device=DEV, max_length=512)
if DEV == "cuda": CE.model.half()
def rerank(cands, pool=50):
    out = []
    for q, c in zip(Q, cands):
        c = list(c[:pool]); s = np.asarray(CE.predict([(q, DOC_TEXT[j]) for j in c], batch_size=64, show_progress_bar=False), dtype=float).reshape(len(c), -1)[:, -1]
        out.append(([c[k] for k in np.argsort(-s)], float(s.max())))
    return out
hyb = [rrf([BM[0][i], dense["medcpt"][0][i]], k=rcfg["fusion"]["rrf_k"], top=DEPTH) for i in range(len(Q))]
t0 = time.perf_counter(); rr = rerank(hyb); rr_ms = (time.perf_counter() - t0) / len(Q) * 1000
RET = {"pipeline": [r[0] for r in rr],
       "bm25_only": [list(BM[0][i]) for i in range(len(Q))], "dense_only": [list(dense["medcpt"][0][i]) for i in range(len(Q))],
       "hybrid_no_rerank": [list(h) for h in hyb],
       "hybrid_bge_base": [r[0] for r in rerank([rrf([BM[0][i], dense["bge-base"][0][i]], k=rcfg["fusion"]["rrf_k"], top=DEPTH) for i in range(len(Q))])],
       "weighted_hybrid": [r[0] for r in rerank([weighted(dense["medcpt"][0][i], dense["medcpt"][1][i], BM[0][i], BM[1][i], 0.6, top=DEPTH) for i in range(len(Q))])]}
REL_HIT = {k: float(np.mean([any(DOC_IDS[j] in {str(d) for d in (dv.get("evidence") or {})} for j in r[:1]) for r, dv in zip(v, [next(x for x in dev if x["id"] == c["id"]) for c in CLAIMS])])) for k, v in RET.items()}
exp.log_result("retrieval", {"rerank_ms_per_claim": round(rr_ms, 1), "top1_is_gold_evidence_doc": REL_HIT}); print("top-1 hits a gold evidence abstract:", REL_HIT)

## 4. End-to-end runs

In [ ]:
QWEN_LABELS = {}
def qwen_labels(pairs, tag):
    "Qwen3-8B labels for (claim, evidence) pairs; cached per (claim, evidence)."
    need = [(c, e) for c, e in pairs if (c, e) not in QWEN_LABELS]
    if need:
        for m_ in (nli.model, CE.model): m_.to("cpu")
        torch.cuda.empty_cache()
        with managed("qwen3-8b") as mc:
            v = LLMVerifier(mc, seed=SEED)
            for c, e in need:
                QWEN_LABELS[(c, e)] = v.verify(e, c, tag=f"verify:e2e:{tag}")[0] or "INS"
        for m_ in (nli.model, CE.model): m_.to(DEV)
    return [QWEN_LABELS[p] for p in pairs]

def run(evidence_ranked, *, verifier="nli", top="top1", calibrated=True, second_opinion=True, abstain=True, tag="pipeline"):
    ev1 = [DOC_TEXT[r[0]] if len(r) else "" for r in evidence_ranked]
    if verifier == "qwen":
        lab = np.array([("SUP" if l == "PART" else l) for l in qwen_labels(list(zip(Q, ev1)), tag)])
        return {"pred": lab, "answered": np.ones(len(Q), bool)}
    if verifier == "zero-shot":
        zs = vcfg["models"]["zero_shot_nli"]; v0 = NLIVerifier(zs["repo"], zs["revision"], device=DEV)
        lab, _ = v0.predict(ev1, Q); del v0; torch.cuda.empty_cache()
        return {"pred": np.array(lab), "answered": np.ones(len(Q), bool)}
    if top == "top3_naive":
        labs = []
        for q, r in zip(Q, evidence_ranked):
            ev = [DOC_TEXT[j] for j in r[:3]]; l3, _ = nli.predict(ev, [q] * len(ev))
            labs.append("SUP" if "SUP" in l3 and "CON" not in l3 else "CON" if "CON" in l3 and "SUP" not in l3 else (l3[0] if ("SUP" in l3) else "INS"))
        return {"pred": np.array(labs), "answered": np.ones(len(Q), bool)}
    z = nli.logits(ev1, Q)
    lab, p, overall = CM.score(Q, ev1, z)
    route_thr = CM.route_threshold
    if not calibrated:
        overall = np.exp(z).max(1) / np.exp(z).sum(1); route_thr = RAW_Q25
    pred = np.array(lab); answered = np.ones(len(Q), bool)
    if second_opinion:
        low = overall < route_thr
        idx = np.where(low)[0]
        if len(idx):
            ql = qwen_labels([(Q[i], ev1[i]) for i in idx], tag)
            for i, l in zip(idx, ql):
                l = "SUP" if l == "PART" else l
                if l != pred[i]:
                    if abstain: answered[i] = False
                    else: pred[i] = l
    elif abstain:
        answered = overall >= CM.abstain_threshold
    return {"pred": pred, "answered": answered}

def macro_f1(g, p, labels=("SUP", "CON", "INS")):
    f = []
    for l in labels:
        tp = np.sum((g == l) & (p == l)); fp = np.sum((g != l) & (p == l)); fn = np.sum((g == l) & (p != l))
        f.append(2 * tp / (2 * tp + fp + fn) if tp else 0.0)
    return float(np.mean(f))
def summarize(name, r):
    a = r["answered"]; g, p = GOLD, r["pred"]
    e2e = np.where(a, p, "ABSTAINED")
    fs_den = max(1, np.sum(np.isin(g[a], ["CON", "INS"])))
    return {"config": name, "coverage": round(float(a.mean()), 3), "macro_F1_answered": round(macro_f1(g[a], p[a]), 4),
            "macro_F1_e2e": round(macro_f1(g, e2e), 4), "FSR_answered": round(float(np.sum((p[a] == "SUP") & np.isin(g[a], ["CON", "INS"])) / fs_den), 4),
            "accuracy_answered": round(float(np.mean(p[a] == g[a])), 4)}
RUNS = {"pipeline": run(RET["pipeline"])}
for k in cfg["ablations"]["retrieval"]:
    RUNS[k] = run(RET[k], tag=k)
RUNS["top3_naive"] = run(RET["pipeline"], top="top3_naive")
RUNS["qwen_only"] = run(RET["pipeline"], verifier="qwen", tag="qwen_only")
RUNS["zero_shot_nli"] = run(RET["pipeline"], verifier="zero-shot")
RUNS["uncalibrated"] = run(RET["pipeline"], calibrated=False, tag="uncal")
RUNS["no_second_opinion"] = run(RET["pipeline"], second_opinion=False)
RUNS["no_abstention"] = run(RET["pipeline"], abstain=False)
RUNS["nli_only_no_conf"] = run(RET["pipeline"], second_opinion=False, abstain=False)
E2E = pd.DataFrame([summarize(k, v) for k, v in RUNS.items()]).set_index("config")
exp.save_table("e2e_ablations", E2E); exp.log_result("e2e_ablations", E2E.reset_index().to_dict(orient="records")); display(E2E)

## 5. H7: where do the errors come from? (oracle-evidence substitution)

In [ ]:
base = RUNS["nli_only_no_conf"]["pred"]                      # the pipeline without abstention / second opinion (pure retrieval + verifier)
oz = nli.logits([c["oracle_text"] for c in CLAIMS], Q); olab = np.array([LAB[k] for k in oz.argmax(1)])
err = base != GOLD
retr = err & (olab == GOLD); verif = err & (olab != GOLD)
rng = np.random.default_rng(SEED); idx = np.arange(len(GOLD)); diffs = []
for _ in range(cfg["n_boot"]):
    s = rng.choice(idx, len(idx)); e = err[s].sum()
    diffs.append((retr[s].sum() - verif[s].sum()) / max(1, e))
lo, hi = np.percentile(diffs, [2.5, 97.5])
H7 = {"errors": int(err.sum()), "retrieval_attributable": int(retr.sum()), "verifier_attributable": int(verif.sum()),
      "retrieval_share": round(float(retr.sum() / max(1, err.sum())), 3), "share_difference_ci": [round(float(lo), 3), round(float(hi), 3)],
      "oracle_evidence_macro_F1": round(macro_f1(GOLD, olab), 4), "retrieved_macro_F1": round(macro_f1(GOLD, base), 4),
      "errors_by_gold": {g: {"retrieval": int((retr & (GOLD == g)).sum()), "verifier": int((verif & (GOLD == g)).sum())} for g in ("SUP", "CON", "INS")}}
H7["supported"] = bool(lo > 0)
exp.log_result("H7", H7); print(json.dumps(H7, indent=1))

## 6. Contextualization ablation (NB04 synthetic context-dependent items)
The NB04 items are rebuilt from the LLM call log (exact replay). Each dependent claim is run through retrieval, top-1 evidence and NLI. The query is the claim as written (A), question + claim (B), or the rewrite (F), with the original claim as the oracle. The verifier reads either the claim as written or the rewrite.

In [ ]:
from cliniver.claims.contextualize import make_context, decontextualize
from cliniver.ingestion.scifact import parse_release as _pr
_, sf_claims = _pr(sf_raw)
items = []
with managed(ccfg["llm_model"]) as mc:
    for c in sf_claims:
        it, _ = make_context(mc, c.text, seed=ccfg["seed"], tag=f"ctx:{c.claim_id}")
        if it is None or not it.valid: continue
        fdep, _ = decontextualize(mc, it.dependent_claim, it.question, it.answer, seed=ccfg["seed"], tag=f"dectx:dependent:{c.claim_id}")
        items.append({"claim": c.text, "dep": it.dependent_claim, "q": it.question, "F": fdep or it.dependent_claim, "labels": c.labels})
    ctx_live = mc.stats
items = [x for x in items if x["labels"] in (("SUPPORT",), ("CONTRADICT",))]
if SMOKE: items = items[:40]
G2 = np.array(["SUP" if x["labels"] == ("SUPPORT",) else "CON" for x in items])
def ctx_eval(queries, verify_text):
    bi = bm.search(queries, DEPTH)[0]
    m = rcfg["encoders"]["medcpt"]; enc = Encoder("medcpt", m["repo"], m["revision"], doc_repo=m["doc_repo"], doc_revision=m["doc_revision"], device=DEV)
    D = torch.from_numpy(np.load(EMB / "medcpt_primary.npy")).to(DEV); di = Encoder.search(enc.encode_queries(queries), D, DEPTH)[0]; enc.unload(); del D; torch.cuda.empty_cache()
    top1 = []
    for q, a, b in zip(queries, bi, di):
        c = list(rrf([a, b], k=rcfg["fusion"]["rrf_k"], top=DEPTH)[:50]); s = np.asarray(CE.predict([(q, DOC_TEXT[j]) for j in c], batch_size=64, show_progress_bar=False), dtype=float).reshape(len(c), -1)[:, -1]
        top1.append(DOC_TEXT[c[int(np.argmax(s))]])
    lab, _ = nli.predict(top1, verify_text)
    return round(macro_f1(G2, np.array(lab), ("SUP", "CON")), 4)
ctx = {"claim_only (verify as written)": ctx_eval([x["dep"] for x in items], [x["dep"] for x in items]),
       "question+claim (verify as written)": ctx_eval([f"{x['q']} {x['dep']}" for x in items], [x["dep"] for x in items]),
       "rewrite F (verify as written)": ctx_eval([x["F"] for x in items], [x["dep"] for x in items]),
       "rewrite F (verify the rewrite)": ctx_eval([x["F"] for x in items], [x["F"] for x in items]),
       "oracle original claim": ctx_eval([x["claim"] for x in items], [x["claim"] for x in items])}
exp.log_result("context_ablation", {"items": len(items), "macro_F1_SUP_CON": ctx, "llm_replay_live_server": ctx_live is not None}); print(len(items), "items", json.dumps(ctx, indent=1))

## 7. Silver check of the real claims (Pillar 3)

In [ ]:
from cliniver.ingestion.qa_datasets import parse_jsonl
from cliniver.evaluation.extraction import NLIJudge
from cliniver.config.llm_config import load_claims_config
clc = load_claims_config()
kqa = parse_jsonl((RAW / "kqa" / SNAP / "questions_w_answers.jsonl").read_bytes(), "kqa_golden", human=True)
med = []
for f in clc.datasets["medlfqa"]["files"]:
    src = f.replace("_test_MedLFQA", "").replace("_wogold", "")
    med += parse_jsonl((RAW / "medlfqa" / SNAP / f"{f}.jsonl").read_bytes(), src, human=False)
QA = {it.qid: it for it in kqa + med}
resp = {json.loads(l)["response_id"]: json.loads(l) for l in (PROC / "responses" / "responses.jsonl").open(encoding="utf-8")}
vc = [json.loads(l) for l in (PROC / "responses" / "verdicts_calibrated.jsonl").open(encoding="utf-8")]
claims_meta = {json.loads(l)["claim_id"]: json.loads(l) for l in (PROC / "responses" / "claims.jsonl").open(encoding="utf-8")}
for m_ in (nli.model, CE.model): m_.to("cpu")
torch.cuda.empty_cache()
judge = NLIJudge(cfg["silver"]["judge"]["repo"], cfg["silver"]["judge"]["revision"], device=DEV)
from transformers import AutoModelForSequenceClassification as _A
lab_idx = {v.lower(): int(k) for k, v in judge.model.config.id2label.items()}
CON_I = next(i for l, i in lab_idx.items() if l.startswith("contra"))
rows = []
for v in vc:
    rid = claims_meta[v["claim_id"]]["response_id"]; it = QA.get(resp[rid]["qid"])
    if it is None: continue
    st = list(it.must_have) + list(it.nice_to_have)
    if not st: continue
    enc = judge.tok(st, [v["claim"]] * len(st), truncation=True, max_length=512, padding=True, return_tensors="pt").to(DEV)
    with torch.inference_mode():
        pr = torch.softmax(judge.model(**enc).logits.float(), -1).cpu().numpy()
    ent, con = pr[:, judge.ent].max(), pr[:, CON_I].max()
    silver = "SUP" if ent >= cfg["silver"]["entail_threshold"] and ent >= con else "CON" if con >= cfg["silver"]["entail_threshold"] else "UNK"
    rows.append({"claim_id": v["claim_id"], "source": "physician (K-QA)" if it.human_statements else "LLM-generated (MedLFQA)", "silver": silver,
                 "final": v["final_verdict"], "generator": v["generator"]})
SV = pd.DataFrame(rows)
tab = pd.crosstab([SV.source, SV.silver], SV.final)
exp.save_table("silver_crosstab", tab); display(tab)
sil = {}
for src, g in SV.groupby("source"):
    s = g[g.silver != "UNK"]
    sil[src] = {"claims": len(g), "silver_labelled": len(s), "silver_SUP": int((s.silver == "SUP").sum()), "silver_CON": int((s.silver == "CON").sum()),
                "pipeline_SUP_given_silver_SUP": round(float((s[s.silver == "SUP"].final == "SUP").mean()), 3) if (s.silver == "SUP").any() else None,
                "pipeline_SUP_given_silver_CON (silver false support)": round(float((s[s.silver == "CON"].final == "SUP").mean()), 3) if (s.silver == "CON").any() else None,
                "pipeline_CON_or_ABST_given_silver_CON": round(float(s[s.silver == "CON"].final.isin(["CON", "ABSTAINED"]).mean()), 3) if (s.silver == "CON").any() else None}
exp.log_result("silver", sil); print(json.dumps(sil, indent=1))
del judge; torch.cuda.empty_cache()
for m_ in (nli.model, CE.model): m_.to(DEV)

## 8. Cost, latency and failure rate of the full pipeline (real claims)

In [ ]:
def log_stats(model, prefix):
    lat, bad = [], 0
    p = LLM_LOG / f"{model}.jsonl"
    for l in p.open(encoding="utf-8"):
        r = json.loads(l)
        if r.get("tag", "").startswith(prefix):
            lat.append(r["latency_s"]); bad += r.get("finish_reason") != "stop"
    return {"calls": len(lat), "median_s": round(float(np.median(lat)), 2) if lat else None, "p95_s": round(float(np.percentile(lat, 95)), 2) if lat else None,
            "not_stopped": bad}
cost = {"answer_generation (NB03)": log_stats("qwen3-8b", "gen:"), "claim_extraction (NB03)": log_stats("qwen3-8b", "claims:"),
        "standalone_rewrite (NB04)": log_stats("qwen3-8b", "dectx:real"), "second_opinion (NB07)": log_stats("qwen3-8b", "verify:real2")}
prev = {k: json.loads((pcfg.paths.absolute("artifacts") / "experiments" / k / "results.json").read_text(encoding="utf-8")) for k in ("EXP-05-001", "EXP-06-001", "EXP-07-001")}
routed_share = prev["EXP-07-001"]["real_claims"]["routed_to_llm"] / prev["EXP-07-001"]["real_claims"]["claims"]
per_claim = {"rewrite_s": cost["standalone_rewrite (NB04)"]["median_s"], "retrieval_p95_s": prev["EXP-06-001"]["real_claims"]["retrieval_latency_p95_s"],
             "second_opinion_expected_s": round(routed_share * (cost["second_opinion (NB07)"]["median_s"] or 0), 2)}
for m_ in (nli.model, CE.model): m_.to("cpu")
torch.cuda.empty_cache()
with managed("qwen3-8b") as mc:   # live measurement of the LLM server (one probe call)
    mc.start(); LLMVerifier(mc, seed=SEED).verify(DOC_TEXT[0], Q[0], tag="verify:e2e:vram_probe"); llm_stats = mc.stats
for m_ in (nli.model, CE.model): m_.to(DEV)
vram = {"llm_qwen3_8b_server": llm_stats, "retrieval_peak_mb": prev["EXP-06-001"]["vram"]["real_claim_retrieval_peak_mb"]}
exp.log_result("cost", {"llm_stage_stats": cost, "per_claim": per_claim, "vram_mb": vram, "routed_share": round(routed_share, 3)})
display(pd.DataFrame(cost).T); print(per_claim, vram)

## 9. Final configuration (pre-registered rule)

In [ ]:
budget_ok = {k: True for k in E2E.index}   # all variants share the ADR-0012 retrieval latency (≤ 0.4 s p95) except the LLM-only verifier
budget_ok["qwen_only"] = False              # ~2–4 s per claim on this laptop (LLM on every claim)
fsr_ref = E2E.loc["pipeline", "FSR_answered"]
cand = E2E[(E2E.FSR_answered <= fsr_ref) & E2E.index.map(lambda k: budget_ok[k])]
best = cand.macro_F1_answered.max()
complexity = {"nli_only_no_conf": 0, "no_second_opinion": 1, "no_abstention": 2, "uncalibrated": 2, "pipeline": 3}
near = [k for k in cand.index if cand.loc[k, "macro_F1_answered"] >= best - 0.01]
FINAL = min(near, key=lambda k: complexity.get(k, 3))
final = {"candidates_meeting_constraints": list(cand.index), "within_0.01_of_best": near, "selected": FINAL, "selected_metrics": E2E.loc[FINAL].to_dict()}
exp.log_result("final_selection", final); print(json.dumps(final, indent=1, default=str))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(E2E.coverage, E2E.macro_F1_answered, c=["C3" if k == FINAL else "C0" for k in E2E.index])
for k, r in E2E.iterrows(): ax.annotate(k, (r.coverage, r.macro_F1_answered), fontsize=7)
ax.set_xlabel("coverage"); ax.set_ylabel("macro-F1 (answered)"); ax.set_title("End-to-end: quality vs coverage (SciFact dev)"); ax.grid(alpha=.3)
exp.save_figure("e2e_tradeoff", fig); plt.show()
exp.finish(); print("results:", exp.dir)

## 10. Discussion
All figures are copied from the outputs above (`artifacts/experiments/EXP-08-001/`). Labels are benchmark (SciFact, human), synthetic (NB04) and silver. **Real-claim accuracy is NOT EVALUATED** (D7).

### 10.1 End-to-end results (300 SciFact dev claims, retrieval over 130,066 abstracts)
| Configuration | Coverage | Macro-F1 (answered) | Macro-F1 (abstained = error) | False-support rate (answered) |
|---|---|---|---|---|
| **Pipeline** (ADR-0011 … 0014) | 0.880 | 0.762 | 0.714 | 0.191 |
| NLI only (no calibration / second opinion / abstention) | 1.000 | 0.726 | **0.726** | 0.188 |
| Uncalibrated routing | 0.847 | **0.773** | 0.707 | 0.175 |
| No second opinion (threshold abstention) | 0.810 | 0.753 | 0.673 | 0.171 |
| No abstention (second opinion kept) | 1.000 | 0.715 | 0.715 | 0.199 |
| Weighted hybrid retrieval | 0.877 | 0.760 | 0.711 | 0.185 |
| BGE-base instead of MedCPT | 0.883 | 0.751 | 0.705 | 0.203 |
| Hybrid without re-ranker | 0.873 | 0.711 | 0.664 | 0.211 |
| MedCPT only | 0.873 | 0.727 | 0.679 | 0.184 |
| BM25 only | 0.883 | 0.699 | 0.653 | 0.181 |
| Qwen3-8B only (no NLI) | 1.000 | 0.718 | 0.718 | 0.250 |
| Zero-shot NLI | 1.000 | 0.675 | 0.675 | 0.153 |
| Top-3 naive aggregation | 1.000 | 0.682 | 0.682 | 0.273 |

- **Components that clearly earn their place:**
  - the **MedCPT re-ranker** (+0.05 macro-F1 over the hybrid without it);
  - the **fine-tuned verifier** (+0.05 over zero-shot NLI; +0.01 over Qwen-only, with far lower false support);
  - **top-1 evidence** (+0.04 over top-3 naive).
- **The confidence layer does not pay for itself in distribution.** It raises answered-claim quality (0.726 → 0.762), but abstains on 12% of claims without lowering false support (0.191 vs 0.188). With abstentions counted as errors it is slightly below plain NLI. Its value lies **out of distribution** (NB07: HealthFC 0.28 → 0.54 with the second opinion).
- The top-1 retrieved abstract is a gold evidence abstract for 48.7% of claims. The re-ranker raises this from 44.3%, and BGE-base matches MedCPT (48.3%).

### 10.2 H7: error attribution
- **H7: NOT SUPPORTED.** Of 81 errors without abstention, **47 (58%) are verifier-attributable** and 34 (42%) retrieval-attributable; the CI of the share difference is [−0.38, 0.05].
- Oracle evidence lifts macro-F1 only from 0.726 to **0.786**, so **the verifier is the main bottleneck**.
- **By gold label:**

  | Gold label | Retrieval errors | Verifier errors |
  |---|---|---|
  | SUP | 7 | **21** |
  | CON | 3 | **13** |
  | INS | **24** | 13 |

  Retrieval errors concentrate on claims for which no evidence exists (INS): a topical but non-addressing abstract is retrieved and then judged SUP or CON.

### 10.3 Contextualization end-to-end (476 synthetic context-dependent items; SUP/CON macro-F1)
| Retrieval query | Verifier reads | Macro-F1 |
|---|---|---|
| claim as written | claim as written | 0.715 |
| question + claim | claim as written | 0.794 |
| rewrite F | claim as written (ADR-0011 policy) | 0.820 |
| **rewrite F** | **the rewrite** | **0.896** |
| original claim (oracle) | original claim | 0.897 |

**Verifying the standalone rewrite instead of the claim as written closes almost the whole gap to the oracle** (+0.076). ADR-0011 kept the original text for verification; this result argues for changing that. The original text stays for display.

### 10.4 Silver check of the real claims (Pillar 3)
- Each real claim is compared with its question's reference statements. Silver labels were assigned to 903 of 2,777 claims.

  | Reference statements | Pipeline SUP when silver SUP | Pipeline SUP when silver CON ("silver false support") | Pipeline CON or ABSTAINED when silver CON |
  |---|---|---|---|
  | Physician (K-QA): 58 silver SUP, 31 silver CON | 0.64 | **0.45** | 0.39 |
  | LLM-generated (MedLFQA): 644 silver SUP, 170 silver CON | 0.70 | **0.47** | 0.29 |

- **Interpretation (silver, noisy):** on real answers, the pipeline agrees with supporting reference statements most of the time, but **labels nearly half of the reference-contradicted claims as SUP**. This is consistent with the benchmark findings: false support and numeric blindness are the dominant risks.

### 10.5 Cost, latency and resources
| Stage | Median | p95 |
|---|---|---|
| Answer generation | 3.65 s | 4.84 s |
| Claim extraction (per answer) | 18.8 s | 30.2 s |
| Standalone rewrite (per claim) | 0.88 s | 1.33 s |
| Retrieval + re-ranking (per claim) | — | 0.37 s |
| Second opinion (35% of claims) | 2.33 s each | ≈ 0.82 s expected per claim |

- No LLM call in these stages ended unfinished.
- GPU memory (live): Qwen3-8B server **4.6 GB** above baseline for one short verification call; NB03 measured **5.3 GB** during generation and claim extraction (same server settings; the peak depends on the workload). Both are within the 6 GB budget. Retrieval 2.4 GB.

### 10.6 Final selection (pre-registered rule)
- The rule selects **"uncalibrated"** routing: the highest answered macro-F1 (0.773) among configurations with false support ≤ the pipeline's, within the latency budget.
- This is a weak preference: it is within noise of the pipeline and has lower end-to-end macro-F1 (0.707).
- The rule optimizes answered-claim quality and does not penalize abstention. That is a limitation of the rule, recorded here rather than changed after seeing the results.

## 11. Limitations
- **No real-claim gold labels** (D7). Silver labels come from NLI against reference statements, which may cover a claim only partially. For most MedLFQA questions the statements are LLM-generated.
- **One end-to-end benchmark** (SciFact dev, 300 claims), so CIs are wide. The synthetic context items favour the rewrite strategy by construction (NB04).
- **Ablations are one-factor-at-a-time**; interactions (e.g. a different verifier with a different retriever) are not explored.
- **The final-selection rule ignores coverage**, as noted in §10.6.
- **Reproducibility incident:** NB03's cache replay had erased its live GPU and latency measurements, because no server started. `ManagedClient.start()` now forces a live measurement; NB03 was re-run to restore them.

## 12. Conclusion and decision
**Conclusion:**
- End to end, CliniVer reaches macro-F1 ≈ 0.71–0.76 on SciFact claims retrieved from 130k abstracts, at ~0.4 s retrieval plus ~1 s LLM time per claim on a laptop.
- **The verifier, not retrieval, is the bottleneck (H7 not supported).**
- **False support is the dominant failure**, on benchmarks (≈ 0.19) and on real claims (silver ≈ 0.45 of contradicted claims).
- Three design changes are clearly indicated by the evidence:
  1. verify the **standalone rewrite** (+0.076 on context-dependent claims);
  2. a verifier trained to catch **numeric / negation / qualifier changes**;
  3. a confidence layer tuned for **end-to-end** quality and out-of-distribution robustness rather than in-distribution abstention.

**Decision (proposed, pending owner approval; ADR-0015):**
- The approved pipeline stands as the **baseline v1**.
- An **optimization iteration** (owner request 2026-11-21) addresses the three changes on **fresh, untouched evaluation data**, so the gains are not tuned on the NB08 results.
- Real-claim accuracy remains NOT EVALUATED until clinical validation.

# Part B — Optimization round 1 (EXP-08-002)
**Owner request 2026-11-21:** optimize the results after NB08. **This part is post-hoc with respect to EXP-08-001**: the weaknesses were found on SciFact dev, so SciFact dev and every set used in NB06–NB08 count as *seen*.

The candidates are trained and selected **without** the two fresh test sets:
1. **SciFact-train test split:** 30% of the SciFact *train* claims, grouped by cited abstract so no abstract is shared across splits, and removed from verifier training.
2. **PUBHEALTH test:** public-health fact-checks (MIT licence), never used before.

Protocol and decision rules: `configs/optimize.yaml`, written before running.

| Lever (from EXP-08-001) | Candidate |
|---|---|
| Verifier is the bottleneck; numeric blindness; false support | **v2 verifier:** v1 recipe + rule perturbations of training SUP pairs (negation / number → CON; hedge removal / population widening → INS; paraphrase → SUP) + **hard negatives** (top re-ranked, non-cited abstracts → INS) |
| Retrieval errors fall on INS claims | **INS threshold** on the top-1 cross-encoder score |
| Numbers | **Numeric guard:** a SUP verdict for a claim containing a number absent from the evidence → INS |
| The confidence layer does not help end to end | **Second-opinion policy:** none / abstain on disagreement (v1) / take the LLM label |
| Context (EXP-08-001 §10.3) | **Verify the standalone rewrite** (policy change) |

- **Baseline:** `v1r` = the v1 recipe retrained on the same training split (no augmentation), with the v1 policy. It has not seen the test claims; the deployed v1 has (it was trained on all SciFact train claims).
- **Selection** (dev split): the highest end-to-end macro-F1 (abstained = error) with false-support rate ≤ the baseline's.

In [ ]:
from cliniver.verification.verifiers import finetune_nli
from cliniver.verification.perturb import PERTURBATIONS
from cliniver.verification.consistency import unsupported_numbers
from cliniver.ingestion.http import RawStore, cached_get
import hashlib, itertools
ocfg = yaml.safe_load((ROOT / "configs" / "optimize.yaml").read_text(encoding="utf-8"))
exp2 = Experiment(exp_id="EXP-08-002-smoke" if SMOKE else "EXP-08-002", title="Optimization round 1 (post-hoc, owner request)",
                  research_question="RQ5–RQ7 optimization", config={"optimize": ocfg}, seed=SEED,
                  packages=["torch", "transformers", "sentence-transformers", "bm25s", "scikit-learn", "pandas", "pyarrow", "cliniver"])
_tr = [json.loads(l) for l in tf.extractfile(next(x for x in tf.getnames() if x.endswith("claims_train.jsonl") and "/._" not in x and "cross" not in x)).read().decode().splitlines() if l.strip()]
# group claims that share a cited / evidence abstract (union-find), then split groups: no abstract crosses splits
parent = {}
def find(x):
    while parent.setdefault(x, x) != x: x = parent[x]
    return x
for c in _tr:
    docs = [f"d{d}" for d in c.get("cited_doc_ids", [])] + [f"d{d}" for d in (c.get("evidence") or {})]
    for d in docs: parent[find(d)] = find(f"c{c['id']}")
    find(f"c{c['id']}")
groups = collections.defaultdict(list)
for c in _tr: groups[find(f"c{c['id']}")].append(c["id"])
gkeys = sorted(groups, key=lambda g: min(groups[g])); random.Random(SEED).shuffle(gkeys)
fr = ocfg["splits"]["scifact_train_claims"]; n_tot = len(_tr); SPLIT = {}; acc = 0
for g in gkeys:
    s = "train" if acc < fr["train"] * n_tot else "dev" if acc < (fr["train"] + fr["dev"]) * n_tot else "test"
    for cid in groups[g]: SPLIT[cid] = s
    acc += len(groups[g])
def claim_items(claims):
    out = []
    for c in claims:
        labs = {s["label"] for d in (c.get("evidence") or {}).values() for s in d}
        g = "SUP" if labs == {"SUPPORT"} else "CON" if labs == {"CONTRADICT"} else "INS" if not labs else None
        if g is not None: out.append({"id": c["id"], "claim": c["claim"], "gold": g})
    return out
SPL_ITEMS = {s: claim_items([c for c in _tr if SPLIT[c["id"]] == s]) for s in ("dev", "test")}
if SMOKE: SPL_ITEMS = {k: v[:30] for k, v in SPL_ITEMS.items()}
split_info = {s: {"claims": sum(v == s for v in SPLIT.values()), "e2e_items": len(SPL_ITEMS.get(s, [])),
                  "gold": dict(Counter(x["gold"] for x in SPL_ITEMS.get(s, [])))} for s in ("train", "dev", "test")}
exp2.log_result("splits", split_info); print(json.dumps(split_info))

## 13. Retrieval for the train-split claims (hard negatives) and the dev / test splits

In [ ]:
def retrieve(queries, pool=50):
    "ADR-0012 retrieval over the NB05 primary corpus: (re-ranked doc indices, their cross-encoder scores) per query."
    bi = bm.search(queries, DEPTH)[0]
    m = rcfg["encoders"]["medcpt"]; enc = Encoder("medcpt", m["repo"], m["revision"], doc_repo=m["doc_repo"], doc_revision=m["doc_revision"], device=DEV)
    D = torch.from_numpy(np.load(EMB / "medcpt_primary.npy")).to(DEV); di = Encoder.search(enc.encode_queries(queries), D, DEPTH)[0]; enc.unload(); del D; torch.cuda.empty_cache()
    out = []
    for q, a, b in zip(queries, bi, di):
        c = list(rrf([a, b], k=rcfg["fusion"]["rrf_k"], top=DEPTH)[:pool])
        s = np.asarray(CE.predict([(q, DOC_TEXT[j]) for j in c], batch_size=64, show_progress_bar=False), dtype=float).reshape(len(c), -1)[:, -1]
        o = np.argsort(-s); out.append(([c[k] for k in o], [float(s[k]) for k in o]))
    return out
tr_claims = [c for c in _tr if SPLIT[c["id"]] == "train"]
if SMOKE: tr_claims = tr_claims[:40]
t0 = time.perf_counter()
R_TR = retrieve([c["claim"] for c in tr_claims])
R_SPL = {s: retrieve([x["claim"] for x in v]) for s, v in SPL_ITEMS.items()}
print(f"retrieval {time.perf_counter() - t0:.0f}s for {len(tr_claims) + sum(len(v) for v in SPL_ITEMS.values())} claims")

## 14. Training data and the two verifiers
Both verifiers use the same base model and hyper-parameters as ADR-0013. The micro-batch is 4 × accumulation 4 (the same effective batch of 16), so the GPU stays within budget (NB06 peaked at 7.1 GB with 8 × 2).

In [ ]:
tr_ids = {c["id"] for c in tr_claims}
sf_tr = [x for x in SETS["scifact_train"] if int(x["id"].split(":")[1]) in tr_ids]
base_pairs = [(x["evidence"], x["claim"], x["label"]) for x in sf_tr + SETS["healthver_train"]]
rng_aug = random.Random(SEED); aug, kinds = [], Counter()
for x in sf_tr + SETS["healthver_train"]:
    if x["label"] != "SUP" or rng_aug.random() >= ocfg["augmentation"]["perturb_fraction"]: continue
    opts = []
    for k, lab in ocfg["augmentation"]["perturbations"].items():
        c2 = PERTURBATIONS[k][0](x["claim"])
        if c2 and c2 != x["claim"]: opts.append((k, c2, lab))
    if opts:
        k, c2, lab = rng_aug.choice(opts); aug.append((x["evidence"], c2, lab)); kinds[k] += 1
hard = []
for c, (ranked, _) in zip(tr_claims, R_TR):
    gold_docs = {str(d) for d in c.get("cited_doc_ids", [])} | {str(d) for d in (c.get("evidence") or {})}
    neg = [j for j in ranked[:ocfg["augmentation"]["hard_negative_depth"]] if DOC_IDS[j] not in gold_docs][:ocfg["augmentation"]["hard_negatives_per_claim"]]
    hard += [(DOC_TEXT[j], c["claim"], "INS") for j in neg]
TRAIN = {"v1r": base_pairs, "v2": base_pairs + aug + hard}
if SMOKE: TRAIN = {k: random.Random(SEED).sample(v, 300) for k, v in TRAIN.items()}
ft = vcfg["finetune"]; fbase = vcfg["models"]["finetune_base"]
for m_ in (nli.model, CE.model): m_.to("cpu")
torch.cuda.empty_cache()
VER, FT_LOG = {}, {}
for name, data in TRAIN.items():
    key = hashlib.sha256(json.dumps([fbase, ft, SEED, name, len(data), sorted(Counter(l for _, _, l in data).items()), SMOKE], sort_keys=True).encode()).hexdigest()[:12]
    d = INTERIM / "models" / f"nli_opt_{name}_{key}"
    t0 = time.perf_counter()
    if (d / "config.json").exists():
        mdl = AutoModelForSequenceClassification.from_pretrained(d); tok = AutoTokenizer.from_pretrained(d)
        hist = json.loads((d / "history.json").read_text(encoding="utf-8")); print("cached", d.name)
    else:
        mdl, tok, hist = finetune_nli(fbase["repo"], fbase["revision"], data, epochs=1 if SMOKE else ft["epochs"], lr=ft["lr"], batch_size=4, grad_accum=4,
                                      max_length=ft["max_length"], seed=SEED, device=DEV)
        d.mkdir(parents=True, exist_ok=True); mdl.save_pretrained(d); tok.save_pretrained(d); (d / "history.json").write_text(json.dumps(hist), encoding="utf-8")
    mdl.to(DEV).eval()
    VER[name] = NLIVerifier(str(d), device=DEV, model=mdl, tokenizer=tok, max_length=ft["max_length"])
    FT_LOG[name] = {"pairs": len(data), "labels": dict(Counter(l for _, _, l in data)), "seconds": round(time.perf_counter() - t0), "history": hist, "dir": d.name}
for m_ in (nli.model, CE.model): m_.to(DEV)
FT_LOG["augmentation"] = {"perturbations": dict(kinds), "hard_negatives": len(hard)}
exp2.log_result("finetune", FT_LOG); print(json.dumps(FT_LOG, indent=1, default=str))
CMS = {"v1": CM, **{k: ConfidenceModel.fit(v, SETS["healthver_dev"]) for k, v in VER.items()}}   # calibration split = HealthVer dev, as in ADR-0014
VERS = {"v1": nli, **VER}

## 15. Scoring every set once
Each set is a list of (verified text, evidence) pairs with gold labels, plus the top-1 cross-encoder score when retrieval is involved. For each verifier we keep its label and its multi-signal confidence; the Qwen3-8B second opinion is computed once for every pair that any verifier routes. The LLM labels depend only on the pair and come from the replay cache when available.

In [ ]:
def scored(claims, evidence, gold, top1_score=None, verifiers=("v1", "v1r", "v2")):
    d = {"Q": list(claims), "ev": list(evidence), "gold": np.array(gold), "top1_score": None if top1_score is None else np.array(top1_score), "lab": {}, "overall": {}}
    for k in verifiers:
        z = VERS[k].logits(d["ev"], d["Q"]); lab, _, ov = CMS[k].score(d["Q"], d["ev"], z)
        d["lab"][k] = np.array(lab); d["overall"][k] = ov
    return d
SCO = {}
for s in ("dev", "test"):
    SCO[s] = scored([x["claim"] for x in SPL_ITEMS[s]], [DOC_TEXT[r[0][0]] for r in R_SPL[s]], [x["gold"] for x in SPL_ITEMS[s]], [r[1][0] for r in R_SPL[s]])
SCO["scifact_dev_seen"] = scored(Q, [DOC_TEXT[r[0]] for r in RET["pipeline"]], GOLD, [r[1] for r in rr])
# PUBHEALTH (fresh): claim vs the fact-checkers' explanation
ph_cfg = ocfg["fresh_tests"]["pubhealth"]
ph_raw = cached_get(RateLimitedClient("datasets", min_interval_s=1.0, max_in_flight=1), RawStore(RAW, "pubhealth", SNAP), "test", "parquet",
                    f"https://huggingface.co/datasets/{ph_cfg['repo']}/resolve/{ph_cfg['revision']}/{ph_cfg['file']}")
PH = pd.read_parquet(io.BytesIO(ph_raw)); PH = PH[PH.label.isin(list(ph_cfg["label_map"]))].reset_index(drop=True)
if SMOKE: PH = PH.sample(60, random_state=SEED).reset_index(drop=True)
SCO["pubhealth"] = scored(PH.claim.tolist(), PH[ph_cfg["evidence_field"]].tolist(), PH.label.map(ph_cfg["label_map"]).tolist())
print({k: len(v["Q"]) for k, v in SCO.items()}, "| PUBHEALTH labels", dict(Counter(SCO["pubhealth"]["gold"])))

In [ ]:
# Real claims (no gold): the verified text is the standalone rewrite (policy change); evidence = the ADR-0012 top-1 unit
from cliniver.retrieval.pipeline import build_units
UNITS, _ = build_units(PROC / "documents.jsonl", PROC / "chunks" / "paragraph-128" / "chunks.jsonl")
uid = {u.unit_id: i for i, u in enumerate(UNITS)}
real = [json.loads(l) for l in (PROC / "responses" / "verdicts.jsonl").open(encoding="utf-8")]
ctxd = {json.loads(l)["claim_id"]: json.loads(l) for l in (PROC / "responses" / "claims_contextualized.jsonl").open(encoding="utf-8")}
if SMOKE: real = real[:40]
r_units = [UNITS[uid[r["evidence"][0]["unit_id"]]] for r in real]
r_text = [ctxd[r["claim_id"]]["standalone_text"] or r["claim"] for r in real]
r_ce = [float(np.asarray(CE.predict([(r["query"], u.index_text)], show_progress_bar=False), dtype=float).reshape(-1)[-1]) for r, u in zip(real, r_units)]
SCO["real"] = scored(r_text, [u.text for u in r_units], ["UNK"] * len(real), r_ce, verifiers=("v1r", "v2"))
print(len(real), "real claims;", sum(t != r["claim"] for t, r in zip(r_text, real)), "verified as their standalone rewrite")

In [ ]:
def routed(d, vk):
    return np.where(d["overall"][vk] < CMS[vk].route_threshold)[0]
need = sorted({(d["Q"][i], d["ev"][i]) for d in SCO.values() for vk in d["lab"] for i in routed(d, vk)})
print(len(need), "unique (claim, evidence) pairs routed to the second opinion;", sum(p not in QWEN_LABELS for p in need), "not yet labelled in this kernel")
for v_ in VER.values(): v_.model.to("cpu")
_ = qwen_labels(need, "opt")      # offloads the v1 verifier and the cross-encoder; one server start
for v_ in VER.values(): v_.model.to(DEV)

## 16. Configuration search on the dev split

In [ ]:
TAU = {q: float(np.quantile(SCO["dev"]["top1_score"], q)) for q in ocfg["search_space"]["ins_threshold_quantile"] if q is not None}
def apply(d, vk, guard, tq, so):
    lab = d["lab"][vk].copy(); ans = np.ones(len(lab), bool)
    if guard:
        for i in np.where(lab == "SUP")[0]:
            if unsupported_numbers(d["Q"][i], d["ev"][i]): lab[i] = "INS"
    if tq is not None and d["top1_score"] is not None:
        lab[d["top1_score"] < TAU[tq]] = "INS"
    if so != "none":
        for i in routed(d, vk):
            l = QWEN_LABELS[(d["Q"][i], d["ev"][i])]; l = "SUP" if l == "PART" else l
            if l != lab[i]:
                if so == "abstain": ans[i] = False
                else: lab[i] = l
    return lab, ans
def metrics(g, p, a):
    e2e = np.where(a, p, "ABSTAINED"); den = max(1, np.sum(np.isin(g[a], ["CON", "INS"])))
    return {"coverage": round(float(a.mean()), 3), "macro_F1_e2e": round(macro_f1(g, e2e), 4), "macro_F1_answered": round(macro_f1(g[a], p[a]), 4),
            "FSR_answered": round(float(np.sum((p[a] == "SUP") & np.isin(g[a], ["CON", "INS"])) / den), 4), "accuracy_e2e": round(float(np.mean(e2e == g)), 4)}
ss = ocfg["search_space"]
GRID = list(itertools.product(ss["verifier"], ss["numeric_guard"], ss["ins_threshold_quantile"], ss["second_opinion"]))
def cname(c): return f"{c[0]}|guard={int(c[1])}|tau={c[2]}|so={c[3]}"
BASE = (ocfg["baseline"]["verifier"], ocfg["baseline"]["numeric_guard"], ocfg["baseline"]["ins_threshold_quantile"], ocfg["baseline"]["second_opinion"])
rows = []
for c in GRID:
    for s in ("dev", "test"):
        p, a = apply(SCO[s], *c); rows.append({"config": cname(c), "split": s, **metrics(SCO[s]["gold"], p, a)})
GR = pd.DataFrame(rows).pivot(index="config", columns="split")
GR.columns = [f"{m}|{s}" for m, s in GR.columns]
fsr_base = GR.loc[cname(BASE), "FSR_answered|dev"]
cand = GR[GR["FSR_answered|dev"] <= fsr_base]
best = cand["macro_F1_e2e|dev"].max()
ncomp = lambda c: int(c[1]) + int(c[2] is not None) + int(c[3] != "none")
near = [c for c in GRID if cname(c) in cand.index and cand.loc[cname(c), "macro_F1_e2e|dev"] >= best - 0.01]
SEL = min(near, key=lambda c: (ncomp(c), -cand.loc[cname(c), "macro_F1_e2e|dev"]))
sel = {"baseline": cname(BASE), "baseline_dev_FSR": fsr_base, "within_0.01_of_best": [cname(c) for c in near], "selected": cname(SEL), "tau_values": TAU}
exp2.save_table("grid", GR); exp2.log_result("selection", sel); print(json.dumps(sel, indent=1))
display(GR.sort_values("macro_F1_e2e|dev", ascending=False).head(15)[[c for c in GR.columns if c.endswith("|dev")]])

## 17. Fresh tests: SciFact-train test split and PUBHEALTH

In [ ]:
def paired_ci(d, ca, cb, n=ocfg["n_boot"]):
    "95% bootstrap CI of (b − a) for end-to-end macro-F1 and FSR(answered), paired over items."
    g = d["gold"]; pa, aa = apply(d, *ca); pb, ab = apply(d, *cb)
    ea, eb = np.where(aa, pa, "ABSTAINED"), np.where(ab, pb, "ABSTAINED")
    rng = np.random.default_rng(SEED); idx = np.arange(len(g)); df, dfs = [], []
    def fsr(p, a, s):
        m = a[s]; den = max(1, np.sum(np.isin(g[s][m], ["CON", "INS"]))); return np.sum((p[s][m] == "SUP") & np.isin(g[s][m], ["CON", "INS"])) / den
    for _ in range(n):
        s = rng.choice(idx, len(idx)); df.append(macro_f1(g[s], eb[s]) - macro_f1(g[s], ea[s])); dfs.append(fsr(pb, ab, s) - fsr(pa, aa, s))
    return {"delta_F1_e2e": round(macro_f1(g, eb) - macro_f1(g, ea), 4), "ci_F1_e2e": [round(float(x), 4) for x in np.percentile(df, [2.5, 97.5])],
            "delta_FSR": round(float(np.mean(dfs)), 4), "ci_FSR": [round(float(x), 4) for x in np.percentile(dfs, [2.5, 97.5])]}
REPORT = {}
for s in ("test", "pubhealth"):
    d = SCO[s]; tab = {}
    for nm, c in {"baseline (v1r, v1 policy)": BASE, "selected": SEL, "v2, v1 policy": ("v2", False, None, "abstain"),
                  "v2, verifier only": ("v2", False, None, "none"), "v1r, verifier only": ("v1r", False, None, "none")}.items():
        p, a = apply(d, *c); tab[nm] = metrics(d["gold"], p, a)
    if s == "pubhealth":
        for nm, c in {"v1 deployed, v1 policy": ("v1", False, None, "abstain"), "v1 deployed, verifier only": ("v1", False, None, "none")}.items():
            p, a = apply(d, *c); tab[nm] = metrics(d["gold"], p, a)
    REPORT[s] = {"table": tab, "selected_vs_baseline": paired_ci(d, BASE, SEL)}
    print(s); display(pd.DataFrame(tab).T); print(REPORT[s]["selected_vs_baseline"])
exp2.log_result("fresh_tests", REPORT)

## 18. Seen sets (secondary): SciFact dev end to end, verifier-level sets, synthetic edits, context items

In [ ]:
seen = {}
d = SCO["scifact_dev_seen"]
seen["scifact_dev_e2e"] = {nm: metrics(d["gold"], *apply(d, *c)) for nm, c in
                          {"v1 deployed, v1 policy (EXP-08-001 pipeline)": ("v1", False, None, "abstain"), "baseline (v1r, v1 policy)": BASE, "selected": SEL}.items()}
lvl = {}
for s in ("healthver_test", "healthfc", "synthetic"):
    X = SETS[s] if not SMOKE else SETS[s][:200]
    g = np.array([x["label"] for x in X]); ev = [x["evidence"] for x in X]; cl = [x["claim"] for x in X]
    for k in ("v1", "v1r", "v2"):
        lab, _ = VERS[k].predict(ev, cl); lab = np.array(lab)
        labs = ("SUP", "CON", "INS") if s != "synthetic" else ("SUP", "CON", "INS", "PART")
        lvl[(s, k)] = {"macro_F1": round(macro_f1(g, lab, labs), 4)}
        if s == "synthetic":
            kind = np.array([x["kind"] for x in X])
            for kk in ("number", "negation"):
                lvl[(s, k)][f"{kk}_edit_caught (CON)"] = round(float(np.mean(lab[kind == kk] == "CON")), 3)
            lvl[(s, k)]["hedge/population_edit_not_SUP"] = round(float(np.mean(lab[np.isin(kind, ["hedge", "population"])] != "SUP")), 3)
            lvl[(s, k)]["paraphrase_kept_SUP"] = round(float(np.mean(lab[kind == "paraphrase"] == "SUP")), 3)
LVL = pd.DataFrame(lvl).T; LVL.index.names = ["set", "verifier"]; display(LVL)
seen["verifier_level"] = {f"{s}|{k}": v for (s, k), v in lvl.items()}
_nli_v1 = nli; nli = VERS[SEL[0]]
seen["context_items_verify_rewrite_selected_verifier"] = ctx_eval([x["F"] for x in items], [x["F"] for x in items])
nli = _nli_v1
exp2.log_result("seen_sets", seen); print(json.dumps(seen["scifact_dev_e2e"], indent=1)); print("context items (rewrite verified):", seen["context_items_verify_rewrite_selected_verifier"])

### 18b. Fairness check of the context-item score (added 2026-11-22, after the first EXP-08-002 run)
- In the first run, the selected verifier scored 0.748 on the NB04 context items (rewrite verified), vs 0.896 for the deployed v1 in EXP-08-001.
- The comparison is confounded:
  - the context items are built from SciFact claims of **both** releases (train and dev);
  - v1 was trained on **all** SciFact train claims, whereas v1r and v2 saw only the train split.
- The items are therefore grouped by training exposure, with the same evidence (the top-1 for the rewrite) for every verifier.
- **Exploratory, post-hoc analysis.**

In [ ]:
tr_all_text = {c["claim"] for c in _tr}
tr_split_text = {c["claim"] for c in _tr if SPLIT[c["id"]] == "train"}
def exposure(t):
    if t not in tr_all_text: return "unseen by all (SciFact dev claims)"
    return "seen by v1, v1r, v2 (train split)" if t in tr_split_text else "seen by v1 only (dev/test splits)"
EXPO = np.array([exposure(x["claim"]) for x in items])
def ctx_top1(queries):
    bi = bm.search(queries, DEPTH)[0]
    m = rcfg["encoders"]["medcpt"]; enc = Encoder("medcpt", m["repo"], m["revision"], doc_repo=m["doc_repo"], doc_revision=m["doc_revision"], device=DEV)
    D = torch.from_numpy(np.load(EMB / "medcpt_primary.npy")).to(DEV); di = Encoder.search(enc.encode_queries(queries), D, DEPTH)[0]; enc.unload(); del D; torch.cuda.empty_cache()
    out = []
    for q, a, b in zip(queries, bi, di):
        c = list(rrf([a, b], k=rcfg["fusion"]["rrf_k"], top=DEPTH)[:50]); s = np.asarray(CE.predict([(q, DOC_TEXT[j]) for j in c], batch_size=64, show_progress_bar=False), dtype=float).reshape(len(c), -1)[:, -1]
        out.append(DOC_TEXT[c[int(np.argmax(s))]])
    return out
F_TOP1 = ctx_top1([x["F"] for x in items]); FQ = [x["F"] for x in items]
fair = {}
for k in ("v1", "v1r", "v2"):
    lab = np.array(VERS[k].predict(F_TOP1, FQ)[0])
    fair[k] = {"all": round(macro_f1(G2, lab, ("SUP", "CON")), 4)}
    for gname in sorted(set(EXPO)):
        msk = EXPO == gname
        fair[k][f"{gname} (n={int(msk.sum())})"] = round(macro_f1(G2[msk], lab[msk], ("SUP", "CON")), 4)
FAIR = pd.DataFrame(fair).T; display(FAIR)
exp2.log_result("context_fairness_check", fair)

## 19. Real claims: re-verification with the selected configuration and the silver check

In [ ]:
d = SCO["real"]; p, a = apply(d, *SEL); FIN = np.where(a, p, "ABSTAINED")
newv = pd.DataFrame({"claim_id": [r["claim_id"] for r in real], "final_v2": FIN})
SV2 = SV.merge(newv, on="claim_id", how="inner")
sil2 = {}
for src, g in SV2.groupby("source"):
    s = g[g.silver != "UNK"]
    def rate(col, sl, vals): x = s[s.silver == sl]; return round(float(x[col].isin(vals).mean()), 3) if len(x) else None
    sil2[src] = {"silver_SUP": int((s.silver == "SUP").sum()), "silver_CON": int((s.silver == "CON").sum()),
                 "v1: SUP|silver SUP": rate("final", "SUP", ["SUP"]), "selected: SUP|silver SUP": rate("final_v2", "SUP", ["SUP"]),
                 "v1: SUP|silver CON (silver false support)": rate("final", "CON", ["SUP"]), "selected: SUP|silver CON (silver false support)": rate("final_v2", "CON", ["SUP"]),
                 "v1: CON or ABST|silver CON": rate("final", "CON", ["CON", "ABSTAINED"]), "selected: CON or ABST|silver CON": rate("final_v2", "CON", ["CON", "ABSTAINED"])}
dist = {"v1 (EXP-07-001)": SV2.final.value_counts(normalize=True).round(3).to_dict(), "selected": pd.Series(FIN).value_counts(normalize=True).round(3).to_dict()}
exp2.log_result("real_claims", {"silver": sil2, "verdict_distribution": dist, "verified_as_rewrite": int(sum(t != r["claim"] for t, r in zip(r_text, real)))})
display(pd.DataFrame(sil2).T); print(dist)
OUT2 = PROC / "responses" / ("verdicts_v2_smoke.jsonl" if SMOKE else "verdicts_v2.jsonl")
ROUTED_SEL = set(routed(d, SEL[0]).tolist()) if SEL[3] != "none" else set()
with OUT2.open("w", encoding="utf-8") as fh:
    for i, r in enumerate(real):
        q = QWEN_LABELS.get((d["Q"][i], d["ev"][i])) if i in ROUTED_SEL else None
        fh.write(json.dumps({"claim_id": r["claim_id"], "claim": r["claim"], "verified_text": d["Q"][i], "decisive_unit": r_units[i].unit_id,
                             "verifier_label": str(d["lab"][SEL[0]][i]), "final_verdict": str(FIN[i]), "overall_confidence": round(float(d["overall"][SEL[0]][i]), 4),
                             "top1_rerank_score": round(r_ce[i], 3), "second_opinion": q, "config": cname(SEL), "corpus_snapshot": SNAP,
                             "evidence_date": r_units[i].pub_date}, ensure_ascii=False) + "\n")
print("wrote", OUT2.name)

## 20. Adoption decision (pre-registered rule)

In [ ]:
t, ph = REPORT["test"], REPORT["pubhealth"]
c1 = t["selected_vs_baseline"]["delta_F1_e2e"] > 0
c2 = t["table"]["selected"]["FSR_answered"] <= t["table"]["baseline (v1r, v1 policy)"]["FSR_answered"] + 0.02
c3 = ph["table"]["selected"]["macro_F1_e2e"] >= ph["table"]["baseline (v1r, v1 policy)"]["macro_F1_e2e"] - 0.02
ADOPT = {"selected": cname(SEL), "e2e_F1_gain_on_fresh_scifact": c1, "fsr_within_0.02": c2, "pubhealth_not_worse_than_0.02": c3, "adopted_as_baseline_v2": bool(c1 and c2 and c3)}
exp2.log_result("adoption", ADOPT); print(json.dumps(ADOPT, indent=1))
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, s in zip(axs, ("test", "pubhealth")):
    tb = pd.DataFrame(REPORT[s]["table"]).T
    ax.barh(tb.index, tb.macro_F1_e2e, color=["C3" if i == "selected" else "C0" for i in tb.index]); ax.set_xlabel("macro-F1 (abstained = error)"); ax.set_title(s); ax.grid(alpha=.3, axis="x")
fig.tight_layout(); exp2.save_figure("fresh_tests", fig); plt.show()
exp2.finish(); print("results:", exp2.dir)

## 24. Discussion: optimization round 1 (EXP-08-002)
All figures come from the outputs above (`artifacts/experiments/EXP-08-002/`).
- **Fresh tests:**
  - the SciFact-train test split: 241 claims, grouped by abstract, never used for training or selection;
  - PUBHEALTH test.
- **Seen sets:** everything used in NB06–NB08.

### 24.1 Selection and adoption (pre-registered rules)
- The dev split selected **`v2 | no numeric guard | no INS threshold | second opinion → take the LLM label`**.
  - All 8 configurations within 0.01 of the best dev score use v2 + "take the LLM label".
  - The rule then picked the one with the fewest components.
- The adoption rule passed on all three criteria, so this configuration is **baseline v2**.

| Fresh test | Baseline (v1 recipe, v1 policy) | Selected | Δ (95% CI) |
|---|---|---|---|
| SciFact-train test, macro-F1 (abstained = error) | 0.700 | **0.755** | **+0.054 [+0.011, +0.099]** |
| SciFact-train test, false-support rate | 0.231 | **0.124** | **−0.106 [−0.168, −0.052]** |
| PUBHEALTH macro-F1 | 0.436 | **0.472** | **+0.036 [+0.020, +0.054]** |
| PUBHEALTH false-support rate | 0.330 | 0.346 | +0.016 [−0.013, +0.045] (no significant change) |

- On the fresh SciFact split, the optimized pipeline is **better and safer at once**: false support is roughly halved.
- On PUBHEALTH it is better, with no significant change in false support. PUBHEALTH stays hard (0.47): consumer fact-checks with journalist explanations as evidence.

### 24.2 Where the gain comes from (component rows, fresh SciFact test)
| Configuration | Coverage | Macro-F1 | False-support rate |
|---|---|---|---|
| v1r verifier only | 1.00 | 0.689 | 0.226 |
| v2 verifier only | 1.00 | 0.672 | **0.088** |
| v2 + v1 policy (abstain on disagreement) | 0.83 | 0.694 (answered 0.771) | 0.087 |
| **v2 + take the LLM label** | 1.00 | **0.755** | 0.124 |

- **v2 is a conservative verifier.** The augmentation adds rule edits plus 811 hard negatives (topical non-cited abstracts → INS). It cuts false support from 0.23 to 0.09, but loses some SUP/CON recall.
- **The LLM second opinion restores that recall** for the low-confidence band. Taking its label beats abstaining.

### 24.3 Robustness to edits (synthetic; the same rule generators as the v2 training augmentation, so **in-generator**)
| Edit | v1 | v2 |
|---|---|---|
| Changed number caught (→ CON) | 0.053 | **0.935** |
| Negation caught (→ CON) | 0.526 | **0.788** |
| Overstated hedge / population not called SUP | 0.247 | **0.847** |
| Paraphrase kept as SUP | 0.820 | 0.757 |

- The number blindness found in NB06 (≤ 12%) is removed for this kind of edit.
- Because the generators are shared with training, this shows the verifier *can* learn the distinction. It does not show that it generalizes to naturally occurring numeric errors. The fresh-test false-support reduction is the stronger evidence.

### 24.4 Costs and regressions (reported with equal weight)
1. **Context items** (§18b, post-hoc fairness check; macro-F1 on SUP/CON, rewrite verified, verifier only):
   - Unseen by all models: v1 0.806, v1r 0.769, **v2 0.682**. The drop is **real, not a training-exposure artifact.**
   - Mechanism: v2 says INS more often, and this metric scores INS as an error even when the retrieved abstract does not address the claim.
   - In the full pipeline, the second opinion compensates (SciFact dev end to end: 0.714 → 0.739).
2. **Real claims, silver check:**
   - Against LLM-generated reference statements, silver false support improves: 0.465 → 0.424.
   - Against **physician** statements it **worsens: 0.452 → 0.548** (31 silver-CON claims, i.e. about 3 claims).
   - Claims labelled CON or ABSTAINED when silver says CON fall from 0.29–0.39 to 0.15–0.19, because abstention no longer exists.
   - On real claims the LLM overrode the verifier for 519 of 2,777 claims (331 INS → SUP).
   - **Taking the LLM label shifts real-claim verdicts towards SUP.** This is the main open risk of baseline v2.
3. **Latency:** 40.4% of real claims are routed to the second opinion (35.4% in v1), about 1 s more per claim.
4. **Unused levers:**
   - The INS threshold on the cross-encoder score is useless: its scores saturate near 1.0 (dev quantiles 0.10 / 0.99 / 0.998).
   - The numeric guard is redundant once v2 catches number edits.

## 25. Limitations (Part B)
- **Post-hoc with respect to EXP-08-001:** the levers were chosen after seeing the NB08 errors. The protection is the fresh test data, which played no part in any choice.
- **One fresh benchmark per kind:** 241 SciFact claims (wide CIs) and PUBHEALTH (verifier-level, given evidence, few INS items: 45).
- **The synthetic edit results are in-generator** (§24.3).
- **Real-claim results are silver** and small for physician statements (31 contradicted claims). Real-claim accuracy remains NOT EVALUATED (D7). The expert-review interface (`cliniver.service`) starts collecting human judgements.
- **One seed, one augmentation recipe;** no hyper-parameter search beyond the pre-registered grid.

## 26. Conclusion and decision (Part B)
- **Conclusion:** on fresh data, round 1 raises end-to-end macro-F1 (+0.054) and **halves false support** on scientific claims. It also improves consumer fact-checks, and makes the verifier sensitive to changed numbers, negations and overstatements.
- **The price:**
  - a more conservative verifier (lower SUP/CON recall when used alone);
  - a pipeline that relies on the LLM second opinion;
  - on physician-referenced real claims, that LLM pushes some contradicted claims to SUP.
- **Decision** (pre-registered rule; ADR-0016): the selected configuration is **baseline v2** and powers the demo service.
- **Round 2 targets the open risk:**
  - let v2's INS/CON act as a veto on LLM SUP overrides, instead of always taking the LLM label;
  - restore SUP/CON recall on context items, e.g. with fewer hard negatives;
  - measure real-claim accuracy with the expert review.

# Part C — Optimization round 1: latency (EXP-08-003)
- **Owner request 2026-11-22:** include latency in the optimization.
- **Protocol** (`configs/optimize_latency.yaml`): written before any latency measurement and before the EXP-08-002 results were seen. The n-gram variant of L1 was added before measurement.
- **Rule:** a lever is adopted only if its quality check passes **and** it cuts its stage's median latency by ≥ 10% on the dev data.
- **Baseline stage latencies (EXP-08-001):**
  - extraction 18.8 s per answer;
  - retrieval p95 0.37 s per claim;
  - second opinion 2.33 s per routed claim;
  - demo: + ~3 s per LLM-server start.

| Lever | Stage | Quality check |
|---|---|---|
| L1 speculative decoding (Qwen3-0.6B draft, or n-gram lookup) | claim extraction | outputs identical, else claim F1 ≥ 0.98 vs baseline; GPU ≤ budget |
| L2 smaller re-rank pool (50 → 30 / 20) | re-ranking | dev end-to-end macro-F1 Δ ≥ −0.01 |
| L3 batched retrieval per answer | retrieval | top-1 evidence identical for ≥ 99% of claims |
| L4 warm LLM server, PyTorch resident | demo overhead | joint GPU peak ≤ budget |
| L5 second-opinion max_tokens 300 → 160 | second opinion | label agreement ≥ 0.98 |

In [ ]:
from cliniver.llm.runtime import LlamaServer, ChatClient, gpu_memory_used_mb
from cliniver.claims.extract import extract_qualified
lat_cfg = yaml.safe_load((ROOT / "configs" / "optimize_latency.yaml").read_text(encoding="utf-8"))
exp3 = Experiment(exp_id="EXP-08-003-smoke" if SMOKE else "EXP-08-003", title="Optimization round 1: latency", research_question="latency (owner request 2026-11-22)",
                  config={"optimize_latency": lat_cfg}, seed=SEED, packages=["torch", "transformers", "sentence-transformers", "bm25s", "cliniver"])
RUN_TS = time.strftime("%Y%m%dT%H%M%S")
LAT_LOG = INTERIM / "llm_calls" / "latency" / RUN_TS          # fresh call logs: every LLM call in Part C is live
BUDGET_MB = lcfg.runtime.vram_budget_gb * 1024
med = lambda x: round(float(np.median(x)), 3) if len(x) else None
p95 = lambda x: round(float(np.percentile(x, 95)), 3) if len(x) else None
def qwen_safe(pairs, tag):
    for v_ in VER.values(): v_.model.to("cpu")
    out = qwen_labels(pairs, tag)
    for v_ in VER.values(): v_.model.to(DEV)
    return out
def sync():
    if DEV == "cuda": torch.cuda.synchronize()
print("latency logs:", LAT_LOG)

## 21. L2: re-rank pool size (SciFact-train dev split, selected EXP-08-002 configuration)

In [ ]:
dq = SCO["dev"]["Q"]
bi = bm.search(dq, DEPTH)[0]
m = rcfg["encoders"]["medcpt"]; enc = Encoder("medcpt", m["repo"], m["revision"], doc_repo=m["doc_repo"], doc_revision=m["doc_revision"], device=DEV)
D = torch.from_numpy(np.load(EMB / "medcpt_primary.npy")).to(DEV); di = Encoder.search(enc.encode_queries(dq), D, DEPTH)[0]; enc.unload(); del D; torch.cuda.empty_cache()
CANDS = [list(rrf([a, b], k=rcfg["fusion"]["rrf_k"], top=DEPTH)) for a, b in zip(bi, di)]
POOL = {}
for pool in lat_cfg["levers"]["L2_rerank_pool"]["values"]:
    lat, top1, s1 = [], [], []
    for q, c in zip(dq, CANDS):
        c = c[:pool]; sync(); t0 = time.perf_counter()
        s = np.asarray(CE.predict([(q, DOC_TEXT[j]) for j in c], batch_size=64, show_progress_bar=False), dtype=float).reshape(len(c), -1)[:, -1]
        sync(); lat.append(time.perf_counter() - t0); k = int(np.argmax(s)); top1.append(c[k]); s1.append(float(s[k]))
    POOL[pool] = {"top1": top1, "score": s1, "lat": lat, "d": scored(dq, [DOC_TEXT[j] for j in top1], SCO["dev"]["gold"], s1, verifiers=(SEL[0],))}
need = sorted({(P["d"]["Q"][i], P["d"]["ev"][i]) for P in POOL.values() for i in routed(P["d"], SEL[0])})
_ = qwen_safe(need, "opt")
L2 = {}
ref = metrics(SCO["dev"]["gold"], *apply(POOL[50]["d"], *SEL))["macro_F1_e2e"]
for pool, P in POOL.items():
    mm = metrics(SCO["dev"]["gold"], *apply(P["d"], *SEL))
    L2[pool] = {"macro_F1_e2e": mm["macro_F1_e2e"], "delta_vs_50": round(mm["macro_F1_e2e"] - ref, 4), "FSR": mm["FSR_answered"],
                "rerank_ms_median": round(1000 * np.median(P["lat"]), 1), "rerank_ms_p95": round(1000 * np.percentile(P["lat"], 95), 1),
                "top1_same_as_pool50": round(float(np.mean(np.array(P["top1"]) == np.array(POOL[50]["top1"]))), 3)}
base_ms = L2[50]["rerank_ms_median"]
ok2 = [p for p in L2 if p != 50 and L2[p]["delta_vs_50"] >= -0.01 and L2[p]["rerank_ms_median"] <= 0.9 * base_ms]
L2_POOL = min(ok2) if ok2 else 50
exp3.log_result("L2_rerank_pool", {"by_pool": L2, "adopted_pool": L2_POOL}); display(pd.DataFrame(L2).T); print("adopted pool:", L2_POOL)

## 22. L1: speculative decoding for claim extraction (and L5 in the same baseline session)
Dev answers: a seeded sample of NB03 answers, plus one untimed warm-up answer per server. Each variant gets a fresh server and a fresh call log, so every call is generated live. The PyTorch models are on the CPU during this section (one large model on the GPU at a time).

In [ ]:
for m_ in [nli.model, CE.model] + [v.model for v in VER.values()]: m_.to("cpu")
torch.cuda.empty_cache(); IDLE_MB = gpu_memory_used_mb()
resp_all = [json.loads(l) for l in (PROC / "responses" / "responses.jsonl").open(encoding="utf-8")]
n_dev = lat_cfg["dev_answers"]; pick = random.Random(SEED).sample(range(len(resp_all)), n_dev + 21)
WARM = resp_all[pick[0]]; DEV_ANS = [resp_all[i] for i in pick[1:1 + n_dev]]; TEST_ANS = [resp_all[i] for i in pick[1 + n_dev:]]
if SMOKE: DEV_ANS, TEST_ANS = DEV_ANS[:3], TEST_ANS[:2]
rt = lcfg.runtime; MK = "qwen3-8b"; L1c = lat_cfg["levers"]["L1_speculative_decoding"]
draft = lambda n: ["--spec-type", "draft-simple", "--model-draft", str(lcfg.model_path(L1c["draft_model"])), "--spec-draft-n-max", str(n), "-ngld", "999",
                   "--cache-type-k-draft", "q8_0", "--cache-type-v-draft", "q8_0"]
VARIANTS = {"baseline": [], **{f"draft06_n{n}": draft(n) for n in L1c["draft_max"]}, "ngram_simple": ["--spec-type", "ngram-simple"]}
so_pairs = [(SCO["dev"]["Q"][i], SCO["dev"]["ev"][i]) for i in routed(SCO["dev"], SEL[0])]
if SMOKE: so_pairs = so_pairs[:5]
EXTRACT, SRV_STATS, SO_RUN = {}, {}, {}
for name, extra in VARIANTS.items():
    time.sleep(2)   # let the previous server release its port
    srv = LlamaServer(ROOT / rt.dir, lcfg.model_path(MK), MK, rt.release, rt.host, rt.port, rt.ctx_size, rt.gpu_layers,
                      extra_args=list(rt.extra_args) + extra, log_path=pcfg.paths.absolute("logs") / f"llama-server_nb08c_{name}.log")
    try:
        srv.__enter__()
        cl = ChatClient(srv, LAT_LOG / f"{name}.jsonl", model_key=MK, model_file=lcfg.models[MK].file, release=rt.release)
        extract_qualified(cl, WARM["answer"], seed=SEED, tag="lat:warmup")
        rows = []
        for a in DEV_ANS:
            ex = extract_qualified(cl, a["answer"], seed=SEED, tag=f"lat:{name}")
            r_ = list(cl.cache.values())[-1]
            rows.append({"response_id": a["response_id"], "latency_s": ex.latency_s, "tokens": ex.completion_tokens, "valid": ex.valid_json,
                         "raw": ex.raw, "claims": [c.text for c in ex.claims], "draft_n": r_.get("draft_n"), "draft_accepted": r_.get("draft_accepted")})
        EXTRACT[name] = rows
        if name == "baseline":     # L5: second opinion with max_tokens 300 vs 160 on the routed dev pairs, same live server
            for mt in (300, 160):
                v = LLMVerifier(cl, seed=SEED, max_tokens=mt); labs, lats = [], []
                for c, e in so_pairs:
                    l, info = v.verify(e, c, tag=f"lat:so{mt}"); labs.append(("SUP" if l == "PART" else l) or "INS"); lats.append(info["latency_s"])
                SO_RUN[mt] = {"labels": labs, "lat": lats}
        SRV_STATS[name] = srv.stats()
    except Exception as exc:   # a variant this build cannot run is recorded, not hidden
        EXTRACT[name] = {"error": f"{type(exc).__name__}: {exc}"}; print(name, "failed:", exc)
    finally:
        srv.__exit__(None, None, None)
print({k: (len(v) if isinstance(v, list) else v) for k, v in EXTRACT.items()})

In [ ]:
from cliniver.evaluation.extraction import NLIJudge
judge = NLIJudge(cfg["silver"]["judge"]["repo"], cfg["silver"]["judge"]["revision"], device=DEV)
_n = lambda t: re.sub(r"\W+", " ", t.lower()).strip()
def matched(A, B):
    "claims of A with a partner in B: identical text, or bidirectional entailment ≥ 0.5 (NB03 judge)"
    n = 0
    for a in A:
        if any(_n(a) == _n(b) for b in B): n += 1; continue
        if not B: continue
        p1 = judge.entail_prob(list(B), [a] * len(B)); p2 = judge.entail_prob([a] * len(B), list(B))
        n += bool(np.any((p1 >= 0.5) & (p2 >= 0.5)))
    return n
base = EXTRACT["baseline"]; b_med = med([r["latency_s"] for r in base]); b_valid = float(np.mean([r["valid"] for r in base]))
L1 = {"baseline": {"median_s": b_med, "p95_s": p95([r["latency_s"] for r in base]), "gpu_mb_above_idle": SRV_STATS["baseline"]["gpu_peak_minus_baseline_mb"]}}
for name, rows in EXTRACT.items():
    if name == "baseline": continue
    if isinstance(rows, dict): L1[name] = rows; continue
    ident = [r["raw"] == b["raw"] for r, b in zip(rows, base)]
    tp_r = sum(len(b["claims"]) if i else matched(b["claims"], r["claims"]) for r, b, i in zip(rows, base, ident))
    tp_p = sum(len(r["claims"]) if i else matched(r["claims"], b["claims"]) for r, b, i in zip(rows, base, ident))
    R = tp_r / max(1, sum(len(b["claims"]) for b in base)); P = tp_p / max(1, sum(len(r["claims"]) for r in rows))
    dn = [r["draft_n"] for r in rows if r["draft_n"]]; da = [r["draft_accepted"] for r in rows if r["draft_n"]]
    L1[name] = {"median_s": med([r["latency_s"] for r in rows]), "p95_s": p95([r["latency_s"] for r in rows]),
                "speedup_median": round(b_med / max(1e-9, med([r["latency_s"] for r in rows])), 2),
                "identical_outputs": round(float(np.mean(ident)), 3), "claim_F1_vs_baseline": round(2 * P * R / max(1e-9, P + R), 4),
                "valid_json": round(float(np.mean([r["valid"] for r in rows])), 3),
                "draft_acceptance": round(sum(da) / sum(dn), 3) if dn else None,
                "gpu_mb_above_idle": SRV_STATS[name]["gpu_peak_minus_baseline_mb"]}
del judge; torch.cuda.empty_cache()
def l1_ok(k):
    x = L1[k]
    return ("error" not in x and (x["identical_outputs"] == 1.0 or x["claim_F1_vs_baseline"] >= 0.98) and x["valid_json"] >= b_valid
            and (x["gpu_mb_above_idle"] or 0) <= BUDGET_MB and x["median_s"] <= 0.9 * b_med)
ok1 = [k for k in L1 if k != "baseline" and l1_ok(k)]
L1_ADOPT = min(ok1, key=lambda k: L1[k]["median_s"]) if ok1 else None
L1_ARGS = VARIANTS[L1_ADOPT] if L1_ADOPT else []
L5 = {"pairs": len(so_pairs)}
if SO_RUN:
    a300, a160 = np.array(SO_RUN[300]["labels"]), np.array(SO_RUN[160]["labels"])
    L5.update({"agreement": round(float(np.mean(a300 == a160)), 3) if len(a300) else None, "median_s_300": med(SO_RUN[300]["lat"]), "median_s_160": med(SO_RUN[160]["lat"])})
    L5["adopted"] = bool(L5["agreement"] is not None and L5["agreement"] >= 0.98 and L5["median_s_160"] <= 0.9 * L5["median_s_300"])
exp3.log_result("L1_speculative_decoding", {"variants": L1, "adopted": L1_ADOPT, "idle_gpu_mb_incl_notebook_context": IDLE_MB})
exp3.log_result("L5_second_opinion_tokens", L5)
display(pd.DataFrame(L1).T); print("L1 adopted:", L1_ADOPT, "| L5:", L5)

## 23. L3, L4 and the end-to-end latency of the service engine
- The **service engine** (`cliniver.service.engine`, the code behind the demo) is loaded with the EXP-08-002 configuration.
- **L3:** batched retrieval for the claims of each dev answer vs one search per claim.
- **L4:** joint GPU peak with the LLM server (adopted L1 flags) running while the PyTorch models stay on the GPU.
- **Final:** per-answer latency on 20 **test** answers, first with the baseline settings, then with all adopted levers. Fresh call logs keep every call live.

In [ ]:
from cliniver.service.engine import Engine, BASELINE_LATENCY
del UNITS, uid; gc.collect()
eng = Engine(log=print, latency=dict(BASELINE_LATENCY), llm_log_dir=LAT_LOG / "engine_baseline"); eng.load()
groups = [r["claims"] for r in EXTRACT["baseline"] if r["claims"]]
t_one, t_many, same1, same3 = [], [], [], []
for g in groups:
    sync(); t0 = time.perf_counter(); one = [eng.retriever.search(c, top=3) for c in g]; sync(); t_one.append(time.perf_counter() - t0)
    sync(); t0 = time.perf_counter(); many = eng.retriever.search_many(g, top=3); sync(); t_many.append(time.perf_counter() - t0)
    same1 += [a[0][0] == b[0][0] for a, b in zip(one, many)]; same3 += [a[0] == b[0] for a, b in zip(one, many)]
L3 = {"answers": len(groups), "claims": len(same1), "per_answer_median_s_per_claim_search": med(t_one), "per_answer_median_s_batched": med(t_many),
      "top1_identical": round(float(np.mean(same1)), 4), "top3_identical": round(float(np.mean(same3)), 4)}
L3["adopted"] = bool(L3["top1_identical"] >= 0.99 and L3["per_answer_median_s_batched"] <= 0.9 * L3["per_answer_median_s_per_claim_search"])
exp3.log_result("L3_batched_retrieval", L3); print(L3)
# L4: LLM server (with the adopted L1 flags) running while the PyTorch models stay on the GPU and do retrieval + NLI work
eng.latency["llm_extra_args"] = L1_ARGS
mc = eng._new_llm().start()
flat = [c for g in groups for c in g][:24]
hits = eng.retriever.search_many(flat, top=1)
_ = eng.nli.logits([eng.units[h[0][0]].text for h in hits], flat)
_ = mc.chat([{"role": "user", "content": "Reply with OK."}], max_tokens=8, tag="lat:l4probe")
time.sleep(1.5); st = mc.stats; mc.__exit__(None, None, None)
L4 = {"joint_gpu_peak_mb": st["gpu_peak_mb"], "idle_gpu_mb_incl_notebook_context": IDLE_MB,
      "joint_above_idle_mb": (st["gpu_peak_mb"] - IDLE_MB) if st["gpu_peak_mb"] and IDLE_MB is not None else None, "budget_mb": BUDGET_MB,
      "server_load_s": st["load_s"]}
L4["adopted"] = bool(L4["joint_above_idle_mb"] is not None and L4["joint_above_idle_mb"] <= BUDGET_MB)
exp3.log_result("L4_warm_server", L4); print(L4)

In [ ]:
ADOPTED = {"llm_extra_args": L1_ARGS, "rerank_pool": L2_POOL, "batched_retrieval": L3["adopted"], "warm_llm": L4["adopted"],
           "so_max_tokens": 160 if L5.get("adopted") else 300}
def run_answers(settings, tag):
    eng.close(); eng.latency.update({**BASELINE_LATENCY, **settings}); eng.retriever.pool = int(eng.latency["rerank_pool"])
    eng.llm_log_dir = LAT_LOG / f"engine_{tag}"
    if eng.latency["warm_llm"]:
        eng._torch_to(eng.dev); eng._warm = eng._new_llm().start()
    out = []
    for a in TEST_ANS:
        t0 = time.perf_counter(); r = eng.verify(a["question"], a["answer"]); wall = time.perf_counter() - t0
        out.append({"response_id": a["response_id"], "wall_s": round(wall, 2), "claims": len(r["claims"]), **r["timings"],
                    "verdicts": [c["verdict"] for c in r["claims"]], "claim_texts": [c["claim"] for c in r["claims"]]})
    eng.close()
    return out
FINAL = {"baseline": run_answers(dict(BASELINE_LATENCY), "baseline"), "adopted": run_answers(ADOPTED, "adopted")}
summ = {}
for k, rows in FINAL.items():
    w = [r["wall_s"] for r in rows]; per_claim = [r["wall_s"] / max(1, r["claims"]) for r in rows]
    summ[k] = {"answers": len(rows), "claims": sum(r["claims"] for r in rows), "per_answer_median_s": med(w), "per_answer_p95_s": p95(w),
               "per_claim_median_s": med(per_claim),
               **{f"{s}_median": med([r.get(s, 0) for r in rows]) for s in ("extraction_and_rewrite_s", "retrieval_and_verification_s", "second_opinion_s")}}
same_claims = [a["claim_texts"] == b["claim_texts"] for a, b in zip(FINAL["baseline"], FINAL["adopted"])]
agree = [x == y for a, b, s in zip(FINAL["baseline"], FINAL["adopted"], same_claims) if s for x, y in zip(a["verdicts"], b["verdicts"])]
summ["agreement"] = {"answers_with_identical_claims": round(float(np.mean(same_claims)), 3), "verdict_agreement_on_those": round(float(np.mean(agree)), 3) if agree else None}
summ["speedup_per_answer_median"] = round(summ["baseline"]["per_answer_median_s"] / max(1e-9, summ["adopted"]["per_answer_median_s"]), 2)
exp3.log_result("end_to_end_latency_test", summ)
exp3.log_result("adoption", {"adopted_settings": ADOPTED, "levers": {"L1": L1_ADOPT, "L2_pool": L2_POOL, "L3": L3["adopted"], "L4": L4["adopted"], "L5": L5.get("adopted")}})
display(pd.DataFrame({k: v for k, v in summ.items() if k in ("baseline", "adopted")}).T); print(json.dumps(summ["agreement"]), "speed-up", summ["speedup_per_answer_median"])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([[r["wall_s"] for r in FINAL["baseline"]], [r["wall_s"] for r in FINAL["adopted"]]]); ax.set_xticks([1, 2], ["baseline settings", "adopted levers"])
ax.set_ylabel("seconds per answer (test answers)"); ax.set_title("End-to-end verification latency (service engine)"); ax.grid(alpha=.3)
exp3.save_figure("latency_end_to_end", fig); plt.show()
exp3.finish(); print("results:", exp3.dir)

## 27. Discussion: latency (EXP-08-003)
All figures are measured live on this laptop (`artifacts/experiments/EXP-08-003/`).

| Lever | Measurement | Quality / budget check | Decision |
|---|---|---|---|
| L1 Qwen3-0.6B draft (8 / 16 tokens) | extraction 20.5 s → 9.96 / 9.85 s (**2.06× / 2.09×**); draft acceptance 0.76 / 0.61 | claim F1 0.978 / 0.990; **GPU 6,489 MB > 6,144 MB budget** | rejected (budget) |
| L1 n-gram speculation | 20.5 s → **11.35 s (1.81×)**; p95 28.0 → 14.8 s | claim F1 **1.000**; 72.5% byte-identical outputs; GPU unchanged (5,339 MB) | **adopted** |
| L2 re-rank pool 50 → 20 | 236 → **95 ms** per claim | dev macro-F1 Δ +0.0002; top-1 unchanged for 90% of claims | **adopted** |
| L3 batched retrieval | 1.75 → 0.90 s per answer (1.95×) | top-1 identical for **98.35%** (< 99% rule) | rejected |
| L4 warm server, PyTorch resident | removes ~3 s per server start | joint GPU peak **7,641 MB** above idle (> budget) | rejected |
| L5 second opinion 160 tokens | 2.69 → 2.69 s | label agreement 0.853 | rejected |

**End to end** (service engine, 20 fresh test answers, 152–154 claims, every call live):

| | Baseline settings | Adopted levers |
|---|---|---|
| Per answer, median | 41.2 s | **30.4 s (1.36×)** |
| Per answer, p95 | 54.8 s | **38.0 s** |
| Extraction + rewrites (median) | 27.8 s | 18.0 s |
| Retrieval + verification (median) | 2.23 s | 1.01 s |
| Second opinion (median) | 11.1 s | 10.6 s |

- **Agreement:** 85% of answers yield identical claims under both settings; on those, 91% of verdicts are identical.
- **Speculative decoding is not bit-exact on this GPU** (its batched verification changes floating-point order), so verdicts can change. The pre-registered checks bound the quality effect: claim F1 1.000; dev macro-F1 unchanged.
- **The remaining cost** is now the second opinion (~11 s per answer; 40% of claims routed) and the LLM-server starts (one or two per answer, ~3 s each). The GPU budget prevents keeping everything resident.

## 28. Limitations, conclusion and decision (Part C)
**Limitations:**
- One laptop and one run per variant; dev sets of 40 answers / 162 claims.
- The GPU figures were measured in the notebook process: they include its CUDA context and are an upper bound for the standalone service.
- The rules are strict by design: L3 (98.35% identical top-1) and the draft model (6.49 GB) missed their thresholds narrowly. They are **not** adopted after the fact.

**Conclusion:**
- Two levers, n-gram speculation and the smaller re-rank pool, make the full verification **1.36× faster per answer** (median 41 → 30 s; p95 55 → 38 s), with the quality checks passed.
- Claim extraction alone is 1.8× faster, and re-ranking 2.5× faster.

**Decision (ADR-0016):** the service uses `--spec-type ngram-simple` and re-rank pool 20. Round-2 latency candidates, each needing its own measurement:
1. fp32 re-ranking, to make batched retrieval bit-stable (L3);
2. a smaller draft context, to bring the draft model within budget;
3. a cheaper second-opinion route, e.g. only for claims the verifier would label SUP.